# Experimento B — Red FHN con control común $u=I=\pm k$

**Hipótesis numérica:** variar la corriente externa común cambia el régimen dinámico de las neuronas (reposo u oscilación), pero **no necesariamente** consigue sincronizar/desincronizar una red.

Las ecuaciones son las mismas (3)–(4), ahora con $I(t)=\pm k$ constante por experimento:

$$\dot v_i=v_i-\frac{v_i^3}{3}-w_i+I+\frac{\lambda}{N}\sum_{j=1}^N A_{ij}(v_i-v_j),\qquad \tau\dot w_i=v_i+a-bw_i.$$

- Fijamos $\lambda=-0.25$ (acoplamiento atractivo con el signo del enunciado); barrido de $I=\pm k$, $k\in[0,1]$ (21 valores por signo).
- Red completa, homogénea y $N=8$, con la misma condición inicial en todas las simulaciones.
- Comparamos $D(t)$, $R(t)$ (coherencia geométrica **orientativa**) y amplitud de $\bar v(t)$ en el tramo final.
- **Advertencia conceptual:** cuando las neuronas convergen a un equilibrio, $R$ carece de interpretación fiable como sincronización de osciladores. La dispersión $D$ sigue siendo válida. Un control común $I(t)$ no rompe por sí solo la sincronización exacta de neuronas idénticas.

Se muestran figuras en el notebook; **no se guardan imágenes**. Instalación: `pip install numpy scipy matplotlib ipython ipywidgets`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    HAY_WIDGETS = True
except ImportError:
    HAY_WIDGETS = False

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.25})

## 1. Modelo, parámetros, red y observables
Todos los cálculos están en este notebook y no dependen de `src/`.

In [ ]:
# === Todas las funciones necesarias están definidas en este notebook ===
# Ecuaciones (3)-(4) del enunciado: el acoplamiento tiene signo +lambda*(v_i-v_j).
PARAM = {"a": 0.7, "b": 0.8, "tau": 12.5}
N = 8
A = np.ones((N, N)) - np.eye(N)  # Grafo completo no dirigido, A_ii = 0.
GRADOS = A.sum(axis=1)

# Mismas condiciones iniciales para TODOS los ensayos (+k, -k y todos los k).
rng = np.random.default_rng(2026)
V0 = rng.uniform(-1.9, 1.4, N)
W0 = rng.uniform(-0.55, 0.75, N)
X0 = np.concatenate([V0, W0])

T_FINAL = 110.0
T_EVAL = np.linspace(0, T_FINAL, 1101)
K_VALORES = np.linspace(0.0, 1.0, 21)


def fhn_red(t, x, lam, corriente, A=A, parametros=PARAM):
    """Ecuaciones (3) y (4) literales, para corrientes escalares o por neurona."""
    n = A.shape[0]
    v, w = x[:n], x[n:]
    a, b, tau = parametros['a'], parametros['b'], parametros['tau']
    grados = A.sum(axis=1)
    acoplamiento = (lam / n) * (grados * v - A @ v)
    dv = v - (v**3)/3 - w + corriente + acoplamiento
    dw = (v + a - b*w) / tau
    return np.concatenate([dv, dw])


def simular(lam, corriente, x0=X0, t_eval=T_EVAL):
    resultado = solve_ivp(
        fhn_red, (float(t_eval[0]), float(t_eval[-1])), x0,
        t_eval=t_eval, args=(lam, corriente), method='DOP853',
        rtol=1e-7, atol=1e-9
    )
    if not resultado.success:
        raise RuntimeError(resultado.message)
    n = len(x0) // 2
    return {"t": resultado.t, "v": resultado.y[:n], "w": resultado.y[n:]}


def equilibrio_individual(corriente):
    """Equilibrio de una sola neurona, con I constante y sin acoplamiento."""
    a, b = PARAM['a'], PARAM['b']
    # Aquí b>0; la raíz es única para 0<b<1.
    v = brentq(lambda q: q - q**3/3 - (q + a)/b + corriente, -10, 10)
    return v, (v+a)/b


def metricas(sol, corriente):
    """D: dispersión de estados; R: coherencia geométrica de fases, orientativa."""
    v, w = sol['v'], sol['w']
    dv, dw = v - v.mean(axis=0), w - w.mean(axis=0)
    D = np.sqrt(np.mean(dv**2 + PARAM['tau'] * dw**2, axis=0))
    veq, weq = equilibrio_individual(corriente)
    ang = np.arctan2(w-weq, v-veq)
    R = np.abs(np.mean(np.exp(1j*ang), axis=0))
    # Amplitud de la señal promedio en el tercio final; un diagnóstico, no una prueba de ciclo.
    inicio = int(0.70 * v.shape[1])
    senal = v.mean(axis=0)[inicio:]
    amplitud = float(senal.max() - senal.min())
    return {"D": D, "R": R, "D_final": float(D[inicio:].mean()),
            "R_final": float(R[inicio:].mean()), "amplitud_media": amplitud}


def curvas_ejemplo(sol, titulo, corriente):
    m = metricas(sol, corriente)
    t, v, w = sol['t'], sol['v'], sol['w']
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for i in range(v.shape[0]):
        axes[0].plot(t, v[i], lw=1.0, alpha=0.85)
    axes[0].set(xlabel='t', ylabel=r'$v_i(t)$', title='Potenciales')
    axes[1].plot(t, m['D'], color='C1', lw=2, label='Dispersión D')
    axes[1].set(xlabel='t', ylabel='D(t)', title='Dispersión entre neuronas')
    axes[2].plot(t, m['R'], color='C2', lw=2)
    axes[2].set(xlabel='t', ylabel='R(t)', ylim=(-.03, 1.03), title='Coherencia geométrica')
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()
    print(f"D promedio (último 30%): {m['D_final']:.4f} | R promedio: {m['R_final']:.4f} | amplitud de media v: {m['amplitud_media']:.3f}")


def animar_plano(sol, titulo='Evolución en espacio de fases', numero_frames=95):
    """Reproductor incrustado en Jupyter: no guarda GIF, MP4 ni imágenes."""
    t, v, w = sol['t'], sol['v'], sol['w']
    n, pasos = v.shape
    indices = np.unique(np.linspace(0, pasos-1, numero_frames).astype(int))
    fig, ax = plt.subplots(figsize=(6, 5))
    # Dibujamos el rastro completo tenue como referencia, y el tramo recorrido sólido.
    for i in range(n):
        ax.plot(v[i], w[i], alpha=.13, lw=1, color=f'C{i%10}')
    trayectorias = [ax.plot([], [], lw=1.5, color=f'C{i%10}')[0] for i in range(n)]
    puntos = [ax.plot([], [], 'o', ms=5, color=f'C{i%10}')[0] for i in range(n)]
    ax.set(xlabel='v', ylabel='w', title=titulo)
    margen_v = .07 * max(1, np.ptp(v)); margen_w = .07 * max(1, np.ptp(w))
    ax.set_xlim(float(v.min()-margen_v), float(v.max()+margen_v))
    ax.set_ylim(float(w.min()-margen_w), float(w.max()+margen_w))
    etiqueta = ax.text(.02, .98, '', va='top', transform=ax.transAxes)

    def actualizar(indice):
        j = int(indices[indice])
        for i in range(n):
            trayectorias[i].set_data(v[i, :j+1], w[i, :j+1])
            puntos[i].set_data([v[i, j]], [w[i, j]])
        etiqueta.set_text(f't = {t[j]:.1f}')
        return (*trayectorias, *puntos, etiqueta)

    pelicula = FuncAnimation(fig, actualizar, frames=len(indices), interval=65, blit=False)
    html = pelicula.to_jshtml(fps=15, default_mode='loop')
    plt.close(fig)
    return HTML(html)

## 2. Barrido numérico de $I=\pm k$

Se fija $\lambda=-0.25$ y se realizan 42 simulaciones. El valor crítico de Hopf para esta parametrización de una **neurona aislada** cerca de $I\approx0.331$ sirve solo como referencia: en la red, el acoplamiento y las perturbaciones pueden afectar los transitorios.

In [ ]:
LAMBDA_FIJA = -0.25
resultados = {}
for signo in (-1, +1):
    for k in K_VALORES:
        corriente = signo*float(k)
        sol = simular(lam=LAMBDA_FIJA, corriente=corriente)
        resultados[(signo, float(k))] = metricas(sol, corriente)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for signo, etiqueta, color in [(-1, r'$I=-k$', 'C0'), (+1, r'$I=+k$', 'C3')]:
    ms = [resultados[(signo, float(k))] for k in K_VALORES]
    axes[0].plot(K_VALORES, [m['D_final'] for m in ms], '-o', ms=3, color=color, label=etiqueta)
    axes[1].plot(K_VALORES, [m['R_final'] for m in ms], '-o', ms=3, color=color, label=etiqueta)
    axes[2].plot(K_VALORES, [m['amplitud_media'] for m in ms], '-o', ms=3, color=color, label=etiqueta)
axes[0].set(title='Sincronización de estados', ylabel='D (promedio final)', xlabel='k')
axes[1].set(title='Coherencia geométrica (orientativa)', ylabel='R (promedio final)', xlabel='k', ylim=(-.03,1.03))
axes[2].set(title='¿Reposo u oscilación?', ylabel='Amplitud de la media de v', xlabel='k')
for ax in axes: ax.legend()
fig.tight_layout(); plt.show()

## 3. Trayectorias representativas
Se comparan corrientes negativas, positivas pequeñas y positivas por encima del umbral del equilibrio individual.

In [ ]:
for corriente in [-0.8, 0.2, 0.6]:
    sol = simular(lam=LAMBDA_FIJA, corriente=corriente)
    curvas_ejemplo(sol, f'I={corriente:+.2f}, lambda={LAMBDA_FIJA:+.2f}', corriente)

## 4. Exploración interactiva del forzamiento común

In [ ]:
def explorar_I(k=0.6, signo='+k'):
    corriente = (-1 if signo.startswith('-') else 1)*float(k)
    sol = simular(lam=LAMBDA_FIJA, corriente=corriente)
    curvas_ejemplo(sol, f'I={corriente:+.2f} | lambda={LAMBDA_FIJA:+.2f}', corriente)

if HAY_WIDGETS:
    interact(explorar_I,
             k=widgets.SelectionSlider(options=np.round(K_VALORES, 2).tolist(), value=0.6, description='k'),
             signo=widgets.Dropdown(options=['+k', '-k'], description='Signo'))
else:
    print('Falta ipywidgets; se muestra un caso fijo. Instala: pip install ipywidgets')
    explorar_I()

## 5. Animación del plano de fases
Reproductor HTML embebido con controles de reproducción. No se generan archivos de imágenes o video.

In [ ]:
sol_animacion = simular(lam=LAMBDA_FIJA, corriente=0.6)
display(animar_plano(sol_animacion, titulo='8 neuronas: I=+0.6, lambda=-0.25'))

## 6. Interpretación y límite de la hipótesis

- $I$ común modifica la dinámica individual (posibilidad de equilibrio estable u oscilaciones). **No equivale** a controlar cada neurona por separado.
- La diagonal sincronizada (todos los $v_i,w_i$ iguales) es invariante: desde sincronización *exacta*, un $I(t)$ común nunca desincroniza neuronas idénticas.
- Con $\lambda$ fijo atractivo, una disminución de $D$ no demuestra que **$I$** haya causado la sincronización por sí solo; habría que compararla con la evolución para otros valores de $\lambda$ o con $\lambda=0$.
- Amplitud de la media $v$ pequeña sugiere reposo, pero en una población desincronizada también podría existir cancelación; revisar las curvas individuales.
- $R$ basado en ángulo geométrico solo es orientativo para ciclos que rodean el equilibrio; no sustituye una definición rigurosa de fase asintótica.